# Trabajo Final — Tópicos en Data Science II

## MLOps Local para Predicción y Monitoreo de Data Drift en Atenciones de Urgencia en Chile

**Magíster en Ciencia de Datos — Universidad de Las Américas**

**Fuente de datos:** Departamento de Estadísticas e Información de Salud (DEIS), Ministerio de Salud de Chile  
**Período de estudio:** 2018–2023  
**Entorno de desarrollo:** Google Colab

### Objetivo general

Desarrollar un pipeline local de Machine Learning que permita modelar el comportamiento de las atenciones de urgencia en Chile y monitorear cambios en la distribución de los datos (*data drift*) a través del tiempo, considerando eventos externos capaces de modificar los patrones históricos de atención.

### Alcance del proyecto

El proyecto analiza las atenciones de urgencia registradas en Chile durante el período
2018–2023, utilizando datos públicos del Departamento de Estadísticas e Información
de Salud (DEIS).

El análisis se centra en la demanda total de atenciones de urgencia, agregada a nivel
semanal, con el propósito de construir un modelo predictivo y evaluar su comportamiento
ante cambios en la distribución de los datos a través del tiempo.

El alcance incluye la preparación y validación de los datos, el modelado predictivo,
la evaluación temporal, el monitoreo de *Data Drift*, el servicio de inferencia mediante
FastAPI y la visualización de resultados mediante Streamlit.


## 1. Configuración del entorno y carga de datos

In [ ]:
# 01
# Carga de librerías

import os
import zipfile
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

### 1.1 Carga de datos


In [ ]:
# 02
# Descarga de las bases originales desde Hugging Face

repo_id = "Alucard08/deis-atenciones-urgencia-raw-2018-2023"

ruta_raw = "/content/data_raw"
os.makedirs(ruta_raw, exist_ok=True)

for anio in range(2018, 2024):

    nombre_archivo = f"AtencionesUrgencia{anio}.zip"

    hf_hub_download(
        repo_id=repo_id,
        filename=nombre_archivo,
        repo_type="dataset",
        local_dir=ruta_raw
    )

    print(f"✅ {nombre_archivo}")

AtencionesUrgencia2018.zip: reconstructing file:   0%|          |  0.00B / 27.9MB            

AtencionesUrgencia2018.zip: downloading bytes:           |  0.00B            

✅ AtencionesUrgencia2018.zip


AtencionesUrgencia2019.zip: reconstructing file:   0%|          |  0.00B / 29.6MB            

AtencionesUrgencia2019.zip: downloading bytes:           |  0.00B            

...

AtencionesUrgencia2022.zip: downloading bytes:           |  0.00B            

✅ AtencionesUrgencia2022.zip


AtencionesUrgencia2023.zip: reconstructing file:   0%|          |  0.00B /  139MB            

AtencionesUrgencia2023.zip: downloading bytes:           |  0.00B            

✅ AtencionesUrgencia2023.zip


In [ ]:
# 03
# Verificación de las bases descargadas desde Hugging Face

archivos_esperados = [
    f"AtencionesUrgencia{anio}.zip"
    for anio in range(2018, 2024)
]

print("Verificación de archivos descargados:\n")

for archivo in archivos_esperados:
    ruta_archivo = os.path.join(ruta_raw, archivo)

    if os.path.exists(ruta_archivo):
        tamaño_mb = os.path.getsize(ruta_archivo) / (1024 ** 2)
        print(f"✅ {archivo} - {tamaño_mb:.1f} MB")
    else:
        print(f"❌ {archivo} NO encontrado")

Verificación de archivos descargados:

✅ AtencionesUrgencia2018.zip - 26.6 MB
✅ AtencionesUrgencia2019.zip - 28.2 MB
✅ AtencionesUrgencia2020.zip - 42.7 MB
✅ AtencionesUrgencia2021.zip - 46.4 MB
✅ AtencionesUrgencia2022.zip - 130.0 MB
✅ AtencionesUrgencia2023.zip - 132.5 MB


## 2. Revisión y comparación de la estructura de los datos

Cada archivo anual descargado desde DEIS se encuentra comprimido en formato ZIP y
contiene dos archivos principales:

- un archivo **CSV**, que contiene los registros de atenciones de urgencia del año;
- un archivo **Excel con el diccionario de datos**, que describe las variables y
  codificaciones utilizadas en la base.

Antes de integrar los distintos años se revisará la estructura real de los archivos CSV,
con el objetivo de identificar posibles cambios en el número de columnas, nombres de
variables, tipos de datos y codificaciones entre períodos.

**Observación sobre el diccionario de datos:**

Durante la revisión se identificó que el diccionario incluido en los distintos archivos
anuales presenta la misma estructura y codificación, incluyendo categorías que no
necesariamente se encuentran presentes en los archivos históricos correspondientes.

Por esta razón, el diccionario será utilizado como referencia para interpretar las
variables, mientras que la disponibilidad real de columnas y códigos será validada
directamente en cada archivo CSV.

### 2.1 Comparación de la estructura de los archivos por año

In [ ]:
# 04

# Revisar estructura de datos

estructura_anual = {}

for anio in range(2018, 2024):

    ruta_zip = os.path.join(
        ruta_raw,
        f"AtencionesUrgencia{anio}.zip"
    )

    with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

        # Buscar el archivo CSV contenido en el ZIP
        archivos_csv = [
            nombre for nombre in archivo_zip.namelist()
            if nombre.lower().endswith(".csv")
        ]

        if len(archivos_csv) == 0:
            print(f"❌ {anio}: no se encontró archivo CSV")
            continue

        nombre_csv = archivos_csv[0]

        # Leer solamente el encabezado
        with archivo_zip.open(nombre_csv) as archivo:
            df_header = pd.read_csv(
                archivo,
                sep=";",
                nrows=0,
                encoding="latin1"
            )

        estructura_anual[anio] = list(df_header.columns)

        print(f"\nAño {anio}")
        print(f"Número de columnas: {len(df_header.columns)}")
        print(df_header.columns.tolist())


Año 2018
Número de columnas: 15
['IdEstablecimiento', 'NEstablecimiento', 'IdCausa', 'GlosaCausa', 'Total', 'Menores_1', 'De_1_a_4', 'De_5_a_14', 'De_15_a_64', 'De_65_y_mas', 'fecha', 'semana', 'GLOSATIPOESTABLECIMIENTO', 'GLOSATIPOATENCION', 'GlosaTipoCampana']

Año 2019
Número de columnas: 15
['IdEstablecimiento', 'NEstablecimiento', 'IdCausa', 'GlosaCausa', 'Total', 'Menores_1', 'De_1_a_4', 'De_5_a_14', 'De_15_a_64', 'De_65_y_mas', 'fecha', 'semana', 'GLOSATIPOESTABLECIMIENTO', 'GLOSATIPOATENCION', 'GlosaTipoCampana']

Año 2020
Número de columnas: 15
['IdEstablecimiento', 'NEstablecimiento', 'IdCausa', 'GlosaCausa', 'Total', 'Menores_1', 'De_1_a_4', 'De_5_a_14', 'De_15_a_64', 'De_65_y_mas', 'fecha', 'semana', 'GLOSATIPOESTABLECIMIENTO', 'GLOSATIPOATENCION', 'GlosaTipoCampana']

Año 2021
Número de columnas: 15
['IdEstablecimiento', 'NEstablecimiento', 'IdCausa', 'GlosaCausa', 'Total', 'Menores_1', 'De_1_a_4', 'De_5_a_14', 'De_15_a_64', 'De_65_y_mas', 'fecha', 'semana', 'GLOSATIPOEST

### Comparación de variables disponibles por año

Se compara la disponibilidad real de variables en cada archivo anual. Esta revisión
permite identificar cambios en la estructura de los datos que podrían afectar su
integración y el funcionamiento posterior del pipeline.

In [ ]:
# 05

# Obtener todas las variables encontradas entre 2018 y 2023
todas_variables = sorted(
    set(
        variable
        for columnas in estructura_anual.values()
        for variable in columnas
    )
)

# Crear tabla de presencia/ausencia
tabla_estructura = pd.DataFrame(index=todas_variables)

for anio in range(2018, 2024):
    tabla_estructura[str(anio)] = [
        "✅" if variable in estructura_anual[anio] else "❌"
        for variable in todas_variables
    ]

tabla_estructura.index.name = "Variable"

display(tabla_estructura)

,2018,2019,2020,2021,2022,2023
Variable,,,,,,
CodigoComuna,❌,❌,❌,❌,❌,✅
CodigoDependencia,❌,❌,❌,❌,❌,✅
CodigoRegion,❌,❌,❌,❌,❌,✅
De_15_a_64,✅,✅,✅,✅,✅,✅
De_1_a_4,✅,✅,✅,✅,✅,✅
De_5_a_14,✅,✅,✅,✅,✅,✅
De_65_y_mas,✅,✅,✅,✅,✅,✅
GLOSATIPOATENCION,✅,✅,✅,✅,✅,✅
GLOSATIPOESTABLECIMIENTO,✅,✅,✅,✅,✅,✅


**Hallazgo:**  
Los archivos correspondientes a 2018–2022 mantienen una estructura homogénea de
15 variables. En 2023 se incorporan seis variables adicionales relacionadas con
región, comuna y dependencia administrativa.

Este cambio evidencia una modificación en el esquema de los datos históricos.
Por el momento, las nuevas variables no serán eliminadas, ya que primero se evaluará
su utilidad para el análisis y el modelado. Para garantizar la compatibilidad temporal,
se identificará posteriormente el conjunto de variables comunes entre todos los años.

### 2.2 Inspección inicial del contenido de los datos

Se revisan registros de distintos períodos para verificar formatos, tipos de datos,
codificaciones y posibles diferencias en la forma en que la información fue registrada.

In [ ]:
# 06

# Se cargará solo una muestra de los datos para optimizar el uso de recursos computacionales

def cargar_muestra(anio, n_filas=5):

    ruta_zip = os.path.join(
        ruta_raw,
        f"AtencionesUrgencia{anio}.zip"
    )

    with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

        archivos_csv = [
            nombre for nombre in archivo_zip.namelist()
            if nombre.lower().endswith(".csv")
        ]

        nombre_csv = archivos_csv[0]

        with archivo_zip.open(nombre_csv) as archivo:

            df_muestra = pd.read_csv(
                archivo,
                sep=";",
                encoding="latin1",
                nrows=n_filas
            )

    return df_muestra

In [ ]:
# 07

muestra_2018 = cargar_muestra(2018)
muestra_2023 = cargar_muestra(2023)

print("Muestra 2018:")
display(muestra_2018)

print("\nMuestra 2023:")
display(muestra_2023)

Muestra 2018:


,IdEstablecimiento,NEstablecimiento,IdCausa,GlosaCausa,Total,Menores_1,De_1_a_4,De_5_a_14,De_15_a_64,De_65_y_mas,fecha,semana,GLOSATIPOESTABLECIMIENTO,GLOSATIPOATENCION,GlosaTipoCampana
0,18-826,SAPU-Hualqui,12,TOTAL CAUSAS SISTEMA CIRCULATORIO,0,0,0,0,0,0,01/01/2018,1,SAPU,Indiferenciado,Ninguna
1,18-826,SAPU-Hualqui,12,TOTAL CAUSAS SISTEMA CIRCULATORIO,0,0,0,0,0,0,02/01/2018,1,SAPU,Indiferenciado,Ninguna
2,18-826,SAPU-Hualqui,12,TOTAL CAUSAS SISTEMA CIRCULATORIO,0,0,0,0,0,0,03/01/2018,1,SAPU,Indiferenciado,Ninguna
3,18-826,SAPU-Hualqui,12,TOTAL CAUSAS SISTEMA CIRCULATORIO,1,0,0,0,0,1,04/01/2018,1,SAPU,Indiferenciado,Ninguna
4,18-826,SAPU-Hualqui,12,TOTAL CAUSAS SISTEMA CIRCULATORIO,0,0,0,0,0,0,05/01/2018,1,SAPU,Indiferenciado,Ninguna



Muestra 2023:


,IdEstablecimiento,NEstablecimiento,IdCausa,GlosaCausa,Total,Menores_1,De_1_a_4,De_5_a_14,De_15_a_64,De_65_y_mas,...,semana,GLOSATIPOESTABLECIMIENTO,GLOSATIPOATENCION,GlosaTipoCampana,CodigoRegion,NombreRegion,CodigoDependencia,NombreDependencia,CodigoComuna,NombreComuna
0,23-985,SAPU Dr. Marcelo Lopetegui Adams,40,"Trastornos neuróticos, trastornos relacionados...",3,0,0,0,2,1,...,27,SAPU,Indiferenciado,Ninguna,10,De Los Lagos,23,Osorno,10301,Osorno
1,21-109,Hospital Dr. Hernán Henríquez Aravena (Temuco),40,"Trastornos neuróticos, trastornos relacionados...",1,0,0,0,1,0,...,27,Hospital,Indiferenciado,Invierno Regiones,9,De La Araucanía,21,Araucanía Sur,9101,Temuco
2,13-920,SUR Los Bajos de San Agustín,40,"Trastornos neuróticos, trastornos relacionados...",1,0,0,0,1,0,...,27,SUR,Indiferenciado,Ninguna,13,Metropolitana de Santiago,13,Metropolitano Sur,13403,Calera de Tango
3,03-910,SAR Alemania,40,"Trastornos neuróticos, trastornos relacionados...",0,0,0,0,0,0,...,27,SAR,Indiferenciado,Ninguna,2,De Antofagasta,3,Antofagasta,2201,Calama
4,16-107,Hospital de Constitución,40,"Trastornos neuróticos, trastornos relacionados...",0,0,0,0,0,0,...,27,Hospital,Indiferenciado,Ninguna,7,Del Maule,16,Del Maule,7102,Constitución


In [ ]:
# 08

print("Tipos de datos 2018:\n")
print(muestra_2018.dtypes)

print("\nTipos de datos 2023:\n")
print(muestra_2023.dtypes)

Tipos de datos 2018:

IdEstablecimiento           object
NEstablecimiento            object
IdCausa                      int64
GlosaCausa                  object
Total                        int64
Menores_1                    int64
De_1_a_4                     int64
De_5_a_14                    int64
De_15_a_64                   int64
De_65_y_mas                  int64
fecha                       object
semana                       int64
GLOSATIPOESTABLECIMIENTO    object
GLOSATIPOATENCION           object
GlosaTipoCampana            object
dtype: object

Tipos de datos 2023:

IdEstablecimiento           object
NEstablecimiento            object
IdCausa                      int64
GlosaCausa                  object
Total                        int64
Menores_1                    int64
De_1_a_4                     int64
De_5_a_14                    int64
De_15_a_64                   int64
De_65_y_mas                  int64
fecha                       object
semana                       in

### 2.3 Revisión de categorías de causa

Antes de integrar los archivos anuales se revisará la variable `IdCausa` y su
correspondiente `GlosaCausa`, con el objetivo de identificar categorías agregadas,
subcategorías diagnósticas y posibles diferencias de codificación entre años.

Esta revisión es necesaria para evitar duplicidades al calcular el volumen de
atenciones de urgencia.

In [ ]:
# 09

def cargar_causas(anio):

    ruta_zip = os.path.join(
        ruta_raw,
        f"AtencionesUrgencia{anio}.zip"
    )

    with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

        archivos_csv = [
            nombre for nombre in archivo_zip.namelist()
            if nombre.lower().endswith(".csv")
        ]

        nombre_csv = archivos_csv[0]

        with archivo_zip.open(nombre_csv) as archivo:

            df_causas = pd.read_csv(
                archivo,
                sep=";",
                encoding="latin1",
                usecols=["IdCausa", "GlosaCausa"]
            )

    causas_unicas = (
        df_causas
        .drop_duplicates()
        .sort_values("IdCausa")
        .reset_index(drop=True)
    )

    return causas_unicas

In [ ]:
# 10

causas_2018 = cargar_causas(2018)
causas_2023 = cargar_causas(2023)

print("Causas disponibles en 2018:")
display(causas_2018)

print("\nCausas disponibles en 2023:")
display(causas_2023)

Causas disponibles en 2018:


,IdCausa,GlosaCausa
0,1,SECCIÓN 1. TOTAL ATENCIONES DE URGENCIA
1,2,TOTAL CAUSAS SISTEMA RESPIRATORIO
2,3,Bronquitis/bronquiolitis aguda (J20-J21)
3,4,Influenza (J09-J11)
4,5,Neumonía (J12-J18)
5,6,"Otra causa respiratoria (J22, J30-J39, J47, J6..."
6,7,CAUSAS SISTEMA RESPIRATORIO
7,8,- LAS DEMÁS CAUSAS
8,10,IRA Alta (J00-J06)
9,11,Crisis obstructiva bronquial (J40-J46)



Causas disponibles en 2023:


,IdCausa,GlosaCausa
0,1,SECCIÓN 1. TOTAL ATENCIONES DE URGENCIA
1,2,TOTAL CAUSAS SISTEMA RESPIRATORIO
2,3,Bronquitis/bronquiolitis aguda (J20-J21)
3,4,Influenza (J09-J11)
4,5,Neumonía (J12-J18)
5,6,"Otra causa respiratoria (J22, J30-J39, J47, J6..."
6,7,CAUSAS SISTEMA RESPIRATORIO
7,8,- LAS DEMÁS CAUSAS
8,10,IRA Alta (J00-J06)
9,11,Crisis obstructiva bronquial (J40-J46)


### 2.4 Verificación de la categoría de total de atenciones

Dado que las bases contienen categorías agregadas y subcategorías diagnósticas,
se verificará que la categoría correspondiente al total de atenciones de urgencia
mantenga la misma codificación durante todo el período 2018–2023.

Esta validación permitirá utilizar una definición homogénea de demanda y evitar
el doble conteo de atenciones.

In [ ]:
# 11

# Verificar la categoría IdCausa = 1 en cada año

for anio in range(2018, 2024):

    causas_anio = cargar_causas(anio)

    total_urgencias = causas_anio[
        causas_anio["IdCausa"] == 1
    ]

    print(f"\nAño {anio}:")
    display(total_urgencias)


Año 2018:


,IdCausa,GlosaCausa
0,1,SECCIÓN 1. TOTAL ATENCIONES DE URGENCIA



Año 2019:


,IdCausa,GlosaCausa
0,1,SECCIÓN 1. TOTAL ATENCIONES DE URGENCIA



Año 2020:


...

,IdCausa,GlosaCausa
0,1,SECCIÓN 1. TOTAL ATENCIONES DE URGENCIA



Año 2022:


,IdCausa,GlosaCausa
0,1,SECCIÓN 1. TOTAL ATENCIONES DE URGENCIA



Año 2023:


,IdCausa,GlosaCausa
0,1,SECCIÓN 1. TOTAL ATENCIONES DE URGENCIA


**Hallazgo:**  
La categoría `IdCausa = 1`, correspondiente a **“SECCIÓN 1. TOTAL ATENCIONES DE URGENCIA”**, se mantiene sin cambios entre 2018 y 2023.

Por esta razón, esta categoría será utilizada como referencia para construir la serie histórica de demanda general de urgencias, evitando sumar categorías diagnósticas que corresponden a subconjuntos del total y que podrían producir doble conteo.

### 2.5 Construcción de la demanda total y verificación de la unidad de observación

Una vez validada la categoría correspondiente al total general de atenciones,
se construirá un subconjunto homogéneo para cada año utilizando `IdCausa = 1`.

Antes de integrar las bases se verificará la unidad de observación
establecimiento-fecha, con el objetivo de identificar registros repetidos que
pudieran producir doble conteo durante la agregación de la demanda.


In [ ]:
# 12

def cargar_total_urgencias(anio):

    ruta_zip = os.path.join(
        ruta_raw,
        f"AtencionesUrgencia{anio}.zip"
    )

    with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

        archivos_csv = [
            nombre for nombre in archivo_zip.namelist()
            if nombre.lower().endswith(".csv")
        ]

        nombre_csv = archivos_csv[0]

        with archivo_zip.open(nombre_csv) as archivo:

            df = pd.read_csv(
                archivo,
                sep=";",
                encoding="latin1",
                usecols=[
                    "IdEstablecimiento",
                    "NEstablecimiento",
                    "IdCausa",
                    "Total",
                    "fecha",
                    "semana",
                    "GLOSATIPOESTABLECIMIENTO",
                    "GLOSATIPOATENCION",
                    "GlosaTipoCampana"
                ]
            )

    # Mantener solamente el total general de atenciones
    df = df[df["IdCausa"] == 1].copy()

    # Registrar el año de origen
    df["anio"] = anio

    return df

In [ ]:
# 13

total_2018 = cargar_total_urgencias(2018)

print("Dimensiones:")
print(total_2018.shape)

print("\nPrimeros registros:")
display(total_2018.head())

Dimensiones:
(166256, 10)

Primeros registros:


,IdEstablecimiento,NEstablecimiento,IdCausa,Total,fecha,semana,GLOSATIPOESTABLECIMIENTO,GLOSATIPOATENCION,GlosaTipoCampana,anio
1827,13-802,SAPU-Recreo,1,124,01/01/2018,1,SAPU,Indiferenciado,Ninguna,2018
1828,13-802,SAPU-Recreo,1,102,02/01/2018,1,SAPU,Indiferenciado,Ninguna,2018
1829,13-802,SAPU-Recreo,1,96,03/01/2018,1,SAPU,Indiferenciado,Ninguna,2018
1830,13-802,SAPU-Recreo,1,70,04/01/2018,1,SAPU,Indiferenciado,Ninguna,2018
1831,13-802,SAPU-Recreo,1,81,05/01/2018,1,SAPU,Indiferenciado,Ninguna,2018


In [ ]:
# 14

# ¿Hay más de una fila para el mismo establecimiento y fecha?
duplicados_2018 = total_2018.duplicated(
    subset=["IdEstablecimiento", "fecha"],
    keep=False
)

print(
    "Registros que comparten establecimiento y fecha:",
    duplicados_2018.sum()
)

Registros que comparten establecimiento y fecha: 732


In [ ]:
# 15

# ¿Hay filas completamente idénticas en todas sus columnas?
duplicados_exactos_2018 = total_2018.duplicated().sum()

print("Duplicados exactos en 2018:", duplicados_exactos_2018)

Duplicados exactos en 2018: 1


In [ ]:
# 16

# Revisar qué diferencia a los registros repetidos
registros_repetidos_2018 = (
    total_2018.loc[
        duplicados_2018,
        [
            "IdEstablecimiento",
            "NEstablecimiento",
            "fecha",
            "Total",
            "GLOSATIPOESTABLECIMIENTO",
            "GLOSATIPOATENCION",
            "GlosaTipoCampana"
        ]
    ]
    .sort_values(["IdEstablecimiento", "fecha"])
)

display(registros_repetidos_2018.head(30))

,IdEstablecimiento,NEstablecimiento,fecha,Total,GLOSATIPOESTABLECIMIENTO,GLOSATIPOATENCION,GlosaTipoCampana
1149672,13-823,SAPU-Laura Vicuña,01/01/2018,192,SAR,Indiferenciado,Ninguna
1149673,13-823,SAR Haydee Lopez,01/01/2018,192,SAR,Indiferenciado,Ninguna
1149734,13-823,SAPU-Laura Vicuña,01/02/2018,120,SAR,Indiferenciado,Ninguna
1149735,13-823,SAR Haydee Lopez,01/02/2018,120,SAR,Indiferenciado,Ninguna
1159457,13-823,SAPU-Laura Vicuña,01/03/2018,133,SAR,Indiferenciado,Ninguna
1159458,13-823,SAR Haydee Lopez,01/03/2018,133,SAR,Indiferenciado,Ninguna
1159519,13-823,SAPU-Laura Vicuña,01/04/2018,283,SAR,Indiferenciado,Ninguna
1159520,13-823,SAR Haydee Lopez,01/04/2018,283,SAR,Indiferenciado,Ninguna
1159579,13-823,SAPU-Laura Vicuña,01/05/2018,215,SAR,Indiferenciado,Ninguna
1159580,13-823,SAR Haydee Lopez,01/05/2018,215,SAR,Indiferenciado,Ninguna


In [ ]:
# 17

# Cuántas filas existen por establecimiento-fecha
conteo_repeticiones_2018 = (
    total_2018
    .groupby(["IdEstablecimiento", "fecha"])
    .size()
    .reset_index(name="n_registros")
    .sort_values("n_registros", ascending=False)
)

display(conteo_repeticiones_2018.head(20))

,IdEstablecimiento,fecha,n_registros
74676,13-823,20/10/2018,2
74675,13-823,20/09/2018,2
74674,13-823,20/08/2018,2
74673,13-823,20/07/2018,2
74672,13-823,20/06/2018,2
74671,13-823,20/05/2018,2
74670,13-823,20/04/2018,2
74669,13-823,20/03/2018,2
74668,13-823,20/02/2018,2
74667,13-823,20/01/2018,2


### 2.6 Investigación de registros repetidos

Durante la verificación de la unidad de observación se detectaron registros que
comparten el mismo identificador de establecimiento y fecha.

Antes de construir la serie temporal se realizará una revisión acotada de estos
casos para determinar si corresponden a duplicados, diferencias de denominación
del establecimiento u otra característica del registro.

El objetivo de esta revisión es evitar el doble conteo de atenciones durante la
agregación de la demanda.

In [ ]:
# 18

# Diagnóstico de registros repetidos antes de la agregación

# 1. Establecimientos involucrados
resumen_repetidos_2018 = (
    total_2018.loc[duplicados_2018]
    .groupby(["IdEstablecimiento", "NEstablecimiento"])
    .size()
    .reset_index(name="n_registros")
    .sort_values("n_registros", ascending=False)
)

print("Establecimientos involucrados en registros repetidos:")
display(resumen_repetidos_2018)


# 2. Duplicados exactos
duplicados_exactos_filas_2018 = total_2018[
    total_2018.duplicated(keep=False)
]

print("\nDuplicados exactos:")
display(duplicados_exactos_filas_2018)


# 3. Comparación de valores Total en registros repetidos
comparacion_repetidos_2018 = (
    total_2018.loc[duplicados_2018]
    .groupby(["IdEstablecimiento", "fecha"])
    .agg(
        n_registros=("Total", "size"),
        n_totales_distintos=("Total", "nunique"),
        total_min=("Total", "min"),
        total_max=("Total", "max"),
        n_nombres=("NEstablecimiento", "nunique")
    )
    .reset_index()
)

print("\nComparación de los registros repetidos:")
display(comparacion_repetidos_2018.head(20))

Establecimientos involucrados en registros repetidos:


,IdEstablecimiento,NEstablecimiento,n_registros
0,13-823,SAPU-Laura Vicuña,365
1,13-823,SAR Haydee Lopez,365
2,18-808,SAPU San Pedro de La Paz,2



Duplicados exactos:


,IdEstablecimiento,NEstablecimiento,IdCausa,Total,fecha,semana,GLOSATIPOESTABLECIMIENTO,GLOSATIPOATENCION,GlosaTipoCampana,anio
1595560,18-808,SAPU San Pedro de La Paz,1,277,31/03/2018,13,SAPU,Indiferenciado,Ninguna,2018
1595561,18-808,SAPU San Pedro de La Paz,1,277,31/03/2018,13,SAPU,Indiferenciado,Ninguna,2018



Comparación de los registros repetidos:


,IdEstablecimiento,fecha,n_registros,n_totales_distintos,total_min,total_max,n_nombres
0,13-823,01/01/2018,2,1,192,192,2
1,13-823,01/02/2018,2,1,120,120,2
2,13-823,01/03/2018,2,1,133,133,2
3,13-823,01/04/2018,2,1,283,283,2
4,13-823,01/05/2018,2,1,215,215,2
5,13-823,01/06/2018,2,1,94,94,2
6,13-823,01/07/2018,2,1,166,166,2
7,13-823,01/08/2018,2,1,149,149,2
8,13-823,01/09/2018,2,1,214,214,2
9,13-823,01/10/2018,2,1,159,159,2


### 2.7 Verificación de duplicidades en el período 2018–2023

Una vez identificado el patrón de registros repetidos en 2018, se verificará de manera
resumida si este comportamiento también está presente en los demás años.

El objetivo es establecer una regla de limpieza homogénea para todo el período,
evitando realizar correcciones particulares para un solo archivo.

In [ ]:
       # 19

# Resumen de duplicidades en el período 2018–2023

resumen_duplicados = []

for anio in range(2018, 2024):

    df_anio = cargar_total_urgencias(anio)

    # Duplicados exactos
    duplicados_exactos = df_anio.duplicated().sum()

    # Grupos con más de un registro para el mismo establecimiento y fecha
    grupos = (
        df_anio
        .groupby(["IdEstablecimiento", "fecha"])
        .agg(
            n_registros=("Total", "size"),
            n_totales_distintos=("Total", "nunique"),
            n_nombres=("NEstablecimiento", "nunique")
        )
        .reset_index()
    )

    grupos_repetidos = grupos[grupos["n_registros"] > 1]

    resumen_duplicados.append({
        "anio": anio,
        "registros": len(df_anio),
        "duplicados_exactos": duplicados_exactos,
        "grupos_repetidos": len(grupos_repetidos),
        "grupos_mismo_total": (
            grupos_repetidos["n_totales_distintos"] == 1
        ).sum(),
        "grupos_total_distinto": (
            grupos_repetidos["n_totales_distintos"] > 1
        ).sum()
    })

resumen_duplicados = pd.DataFrame(resumen_duplicados)

display(resumen_duplicados)

,anio,registros,duplicados_exactos,grupos_repetidos,grupos_mismo_total,grupos_total_distinto
0,2018,166256,1,366,366,0
1,2019,168551,0,365,365,0
2,2020,194541,0,0,0,0
3,2021,220406,0,0,0,0
4,2022,223158,0,0,0,0
5,2023,222477,1,1,1,0


### 2.8 Limpieza e integración de las bases 2018–2023

La revisión de duplicidades mostró que los registros repetidos para una misma
combinación de establecimiento y fecha presentan el mismo valor de `Total`.
Por esta razón, se conservará un único registro por establecimiento, fecha y
valor de demanda.

Posteriormente, las bases anuales serán integradas en un único dataset para
construir la serie temporal de atenciones de urgencia.

In [ ]:
# 20
# Limpieza e integración de las bases 2018–2023

bases_limpias = []
resumen_limpieza = []

for anio in range(2018, 2024):

    df_anio = cargar_total_urgencias(anio).copy()

    registros_iniciales = len(df_anio)

    # Convertir fecha a formato datetime
    df_anio["fecha"] = pd.to_datetime(
        df_anio["fecha"],
        format="%d/%m/%Y",
        errors="coerce"
    )

    fechas_invalidas = df_anio["fecha"].isna().sum()

    # Eliminar duplicados exactos
    df_anio = df_anio.drop_duplicates()

    # Eliminar registros repetidos con mismo establecimiento,
    # fecha y valor de Total
    df_anio = (
        df_anio
        .sort_values(["IdEstablecimiento", "fecha"])
        .drop_duplicates(
            subset=["IdEstablecimiento", "fecha", "Total"],
            keep="first"
        )
    )

    registros_finales = len(df_anio)

    resumen_limpieza.append({
        "anio": anio,
        "registros_iniciales": registros_iniciales,
        "registros_finales": registros_finales,
        "registros_eliminados": registros_iniciales - registros_finales,
        "fechas_invalidas": fechas_invalidas
    })

    bases_limpias.append(df_anio)


# Integrar todos los años
df_urgencias = pd.concat(
    bases_limpias,
    ignore_index=True
)

# Ordenar cronológicamente
df_urgencias = df_urgencias.sort_values(
    ["fecha", "IdEstablecimiento"]
).reset_index(drop=True)


# Resumen de la limpieza
resumen_limpieza = pd.DataFrame(resumen_limpieza)

display(resumen_limpieza)

print("\nDimensiones del dataset integrado:")
print(df_urgencias.shape)

print("\nPeríodo disponible:")
print(df_urgencias["fecha"].min(), "→", df_urgencias["fecha"].max())

,anio,registros_iniciales,registros_finales,registros_eliminados,fechas_invalidas
0,2018,166256,165890,366,0
1,2019,168551,168186,365,0
2,2020,194541,194541,0,0
3,2021,220406,220406,0,0
4,2022,223158,223158,0,0
5,2023,222477,222476,1,0



Dimensiones del dataset integrado:
(1194657, 10)

Período disponible:
2018-01-01 00:00:00 → 2023-12-31 00:00:00


**Síntesis de calidad de los datos:**

La revisión de las bases 2018–2023 permitió identificar cambios de esquema entre
períodos, registros duplicados y diferencias en la denominación de algunos
establecimientos.

Se eliminaron 732 registros repetidos cuya inclusión podía producir doble conteo
en la demanda agregada. No se identificaron fechas inválidas, valores nulos en las
variables principales ni valores negativos en el total de atenciones.

Estas validaciones se realizaron antes de construir la serie temporal para evitar
que problemas de calidad de datos fueran interpretados posteriormente como cambios
reales en la distribución o como Data Drift.


### 2.9 Validación del dataset integrado y construcción de la serie temporal

Una vez finalizada la limpieza e integración de las bases 2018–2023, se realizará una
validación final de las variables principales para comprobar la ausencia de valores
nulos, fechas inválidas y valores negativos en la demanda.

Posteriormente, los registros serán agregados temporalmente para construir una serie
nacional de atenciones de urgencia. Se analizará inicialmente la demanda diaria y luego
se generará una serie semanal, con el objetivo de reducir variaciones de corto plazo y
facilitar la identificación de cambios estructurales en el comportamiento de la demanda.

In [ ]:
# 21
# Validación rápida del dataset integrado

columnas_clave = [
    "IdEstablecimiento",
    "NEstablecimiento",
    "Total",
    "fecha",
    "semana"
]

print("Valores nulos en variables clave:\n")
display(df_urgencias[columnas_clave].isnull().sum().to_frame("nulos"))

print("\nValores negativos en Total:")
print((df_urgencias["Total"] < 0).sum())

print("\nEstadísticos básicos de Total:")
display(df_urgencias["Total"].describe())

In [ ]:
# 22
# Construcción de la serie temporal nacional de urgencias

serie_diaria = (
    df_urgencias
    .groupby("fecha")
    .agg(
        atenciones=("Total", "sum"),
        establecimientos=("IdEstablecimiento", "nunique")
    )
    .reset_index()
    .sort_values("fecha")
)

print("Serie diaria:")
print(serie_diaria.shape)

print("\nPrimeros registros:")
display(serie_diaria.head())

print("\nÚltimos registros:")
display(serie_diaria.tail())

Serie diaria:
(2191, 3)

Primeros registros:


,fecha,atenciones,establecimientos
0,2018-01-01,43291,454
1,2018-01-02,50846,455
2,2018-01-03,46935,451
3,2018-01-04,44452,451
4,2018-01-05,43029,452



Últimos registros:


,fecha,atenciones,establecimientos
2186,2023-12-27,53292,603
2187,2023-12-28,51472,607
2188,2023-12-29,49171,599
2189,2023-12-30,52800,603
2190,2023-12-31,43124,604


In [ ]:
# 23
# Agregación semanal de la demanda nacional

serie_semanal = (
    serie_diaria
    .set_index("fecha")
    .resample("W-SUN")
    .agg(
        atenciones=("atenciones", "sum"),
        establecimientos=("establecimientos", "mean")
    )
    .reset_index()
)

serie_semanal["establecimientos"] = (
    serie_semanal["establecimientos"].round(1)
)

print("Dimensiones de la serie semanal:")
print(serie_semanal.shape)

display(serie_semanal.head())

Dimensiones de la serie semanal:
(313, 3)


,fecha,atenciones,establecimientos
0,2018-01-07,321162,453.3
1,2018-01-14,308182,454.3
2,2018-01-21,304142,454.1
3,2018-01-28,296492,454.4
4,2018-02-04,287812,453.6


In [ ]:
# 24
# Curva histórica semanal de atenciones de urgencia

import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.plot(
    serie_semanal["fecha"],
    serie_semanal["atenciones"]
)

plt.axvline(
    pd.Timestamp("2020-03-01"),
    linestyle="--",
    label="Inicio período pandemia"
)

plt.title(
    "Atenciones de urgencia semanales en Chile (2018–2023)"
)
plt.xlabel("Fecha")
plt.ylabel("Número de atenciones")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

**Hallazgo:**  
La serie histórica muestra una disminución abrupta de las atenciones de urgencia durante
el inicio de 2020, seguida de una recuperación progresiva durante 2021 y niveles más
elevados y variables durante 2022–2023.

Este comportamiento evidencia un cambio importante respecto del patrón observado en
el período previo, lo que justifica el uso de este caso para evaluar técnicas de
monitoreo de *data drift*.

## 3. Preparación para el modelado

El objetivo predictivo será estimar el número de atenciones de urgencia de una semana
utilizando exclusivamente información disponible en semanas anteriores.

Dado que los datos poseen una estructura temporal, no se utilizará una división aleatoria
de entrenamiento y prueba. El modelo será entrenado con el período prepandemia
(2018–2019) y posteriormente será evaluado sobre ventanas temporales sucesivas
correspondientes a 2020–2023.

Esta estrategia permite simular un escenario real de producción y analizar cómo se
comporta el modelo cuando la distribución de los datos cambia con el tiempo.

In [ ]:
# 25
# Creación de variables temporales para el modelo

df_modelo = serie_semanal.copy()

# Variables de calendario
df_modelo["semana_anio"] = df_modelo["fecha"].dt.isocalendar().week.astype(int)
df_modelo["mes"] = df_modelo["fecha"].dt.month
df_modelo["anio"] = df_modelo["fecha"].dt.year

# Valores históricos de demanda
df_modelo["lag_1"] = df_modelo["atenciones"].shift(1)
df_modelo["lag_2"] = df_modelo["atenciones"].shift(2)
df_modelo["lag_4"] = df_modelo["atenciones"].shift(4)

# Promedio de las cuatro semanas anteriores
df_modelo["media_4_sem"] = (
    df_modelo["atenciones"]
    .shift(1)
    .rolling(window=4)
    .mean()
)

# Eliminar semanas iniciales sin historia suficiente
df_modelo = df_modelo.dropna().reset_index(drop=True)

print("Dimensiones del dataset para modelado:")
print(df_modelo.shape)

display(df_modelo.head())

Dimensiones del dataset para modelado:
(309, 10)


,fecha,atenciones,establecimientos,semana_anio,mes,anio,lag_1,lag_2,lag_4,media_4_sem
0,2018-02-04,287812,453.6,5,2,2018,296492.0,304142.0,321162.0,307494.50
1,2018-02-11,289620,454.7,6,2,2018,287812.0,296492.0,308182.0,299157.00
2,2018-02-18,294329,454.1,7,2,2018,289620.0,287812.0,304142.0,294516.50
3,2018-02-25,301090,453.6,8,2,2018,294329.0,289620.0,296492.0,292063.25
4,2018-03-04,295629,453.7,9,3,2018,301090.0,294329.0,287812.0,293212.75


### 3.1 Prevención de fuga de información

Las variables temporales fueron construidas utilizando únicamente información previa
a la semana que se desea predecir.

Las variables `lag_1`, `lag_2`, `lag_4` y `media_4_sem` representan demanda histórica,
por lo que el valor real de la semana objetivo no es utilizado como predictor.
Esto evita introducir información futura en el entrenamiento (*data leakage*).

In [ ]:
# 26
# Definición de variables predictoras y variable objetivo

features = [
    "semana_anio",
    "mes",
    "lag_1",
    "lag_2",
    "lag_4",
    "media_4_sem"
]

target = "atenciones"

X = df_modelo[features]
y = df_modelo[target]

print("Variables predictoras:")
print(features)

print("\nVariable objetivo:")
print(target)

Variables predictoras:
['semana_anio', 'mes', 'lag_1', 'lag_2', 'lag_4', 'media_4_sem']

Variable objetivo:
atenciones


In [ ]:
# 27
# División temporal entre entrenamiento y producción simulada

train = df_modelo[df_modelo["anio"] <= 2019].copy()

prod_2020 = df_modelo[df_modelo["anio"] == 2020].copy()
prod_2021 = df_modelo[df_modelo["anio"] == 2021].copy()
prod_2022 = df_modelo[df_modelo["anio"] == 2022].copy()
prod_2023 = df_modelo[df_modelo["anio"] == 2023].copy()

print("Distribución temporal:\n")

print("Entrenamiento 2018–2019:", train.shape)
print("Producción 2020:", prod_2020.shape)
print("Producción 2021:", prod_2021.shape)
print("Producción 2022:", prod_2022.shape)
print("Producción 2023:", prod_2023.shape)

Distribución temporal:

Entrenamiento 2018–2019: (100, 10)
Producción 2020: (52, 10)
Producción 2021: (52, 10)
Producción 2022: (52, 10)
Producción 2023: (53, 10)


### 3.2 Selección del modelo mediante validación temporal

Antes de evaluar el comportamiento del modelo en los años de producción simulada,
se realizará una validación exclusivamente sobre el período de entrenamiento
2018–2019.

Debido a la naturaleza temporal de los datos se utilizará `TimeSeriesSplit`,
manteniendo siempre el orden cronológico. Se comparará además el desempeño de los
modelos con un baseline ingenuo que asume que la demanda de la próxima semana será
igual a la observada en la semana anterior.

In [ ]:
# 28
# Validación temporal y comparación de modelos

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.base import clone

X_train = train[features]
y_train = train[target]

# Validación temporal
tscv = TimeSeriesSplit(n_splits=5)

# Modelos candidatos
modelos = {
    "Regresión Lineal": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LinearRegression())
    ]),

    "Random Forest": Pipeline([
        ("modelo", RandomForestRegressor(
            n_estimators=200,
            max_depth=5,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("modelo", GradientBoostingRegressor(
            random_state=42
        ))
    ])
}

resultados_cv = []

for fold, (idx_train, idx_val) in enumerate(tscv.split(X_train), start=1):

    X_tr = X_train.iloc[idx_train]
    X_val = X_train.iloc[idx_val]

    y_tr = y_train.iloc[idx_train]
    y_val = y_train.iloc[idx_val]

    # Baseline: próxima semana = semana anterior
    pred_baseline = X_val["lag_1"]

    resultados_cv.append({
        "modelo": "Baseline lag_1",
        "fold": fold,
        "MAE": mean_absolute_error(y_val, pred_baseline),
        "RMSE": np.sqrt(mean_squared_error(y_val, pred_baseline))
    })

    # Modelos de Machine Learning
    for nombre, modelo in modelos.items():

        modelo_fold = clone(modelo)

        modelo_fold.fit(X_tr, y_tr)

        pred = modelo_fold.predict(X_val)

        resultados_cv.append({
            "modelo": nombre,
            "fold": fold,
            "MAE": mean_absolute_error(y_val, pred),
            "RMSE": np.sqrt(mean_squared_error(y_val, pred))
        })

resultados_cv = pd.DataFrame(resultados_cv)

resumen_cv = (
    resultados_cv
    .groupby("modelo")
    .agg(
        MAE_promedio=("MAE", "mean"),
        RMSE_promedio=("RMSE", "mean")
    )
    .sort_values("MAE_promedio")
    .reset_index()
)

display(resumen_cv)

,modelo,MAE_promedio,RMSE_promedio
0,Baseline lag_1,13377.762500,17546.477780
1,Regresión Lineal,15962.372886,19970.065540
2,Random Forest,16906.702109,21044.901925
3,Gradient Boosting,19009.626787,23209.159566


### 3.3 Entrenamiento del modelo seleccionado y evaluación en producción

El modelo de Machine Learning con menor error promedio durante la validación temporal
será entrenado utilizando todo el período de referencia 2018–2019.

Posteriormente se evaluará de forma independiente en las ventanas de producción
2020, 2021, 2022 y 2023, permitiendo observar la evolución de su desempeño ante
cambios en el comportamiento de la demanda.

In [ ]:
# 29
# Selección del mejor modelo y evaluación en producción

# Seleccionar solamente modelos ML
resultados_ml = resumen_cv[
    resumen_cv["modelo"] != "Baseline lag_1"
].copy()

mejor_nombre = resultados_ml.iloc[0]["modelo"]

# Recuperar y entrenar el mejor modelo
mejor_modelo = clone(modelos[mejor_nombre])

mejor_modelo.fit(
    train[features],
    train[target]
)

print("Modelo seleccionado:")
print(mejor_nombre)

# Ventanas de producción
ventanas = {
    2020: prod_2020,
    2021: prod_2021,
    2022: prod_2022,
    2023: prod_2023
}

metricas_produccion = []
predicciones_produccion = []

for anio, datos in ventanas.items():

    X_prod = datos[features]
    y_real = datos[target]

    # Predicción del modelo
    y_pred = mejor_modelo.predict(X_prod)

    # Baseline
    y_baseline = datos["lag_1"]

    metricas_produccion.append({
        "anio": anio,

        "MAE_modelo": mean_absolute_error(
            y_real,
            y_pred
        ),

        "RMSE_modelo": np.sqrt(
            mean_squared_error(
                y_real,
                y_pred
            )
        ),

        "MAE_baseline": mean_absolute_error(
            y_real,
            y_baseline
        )
    })

    # Guardar predicciones para análisis posterior
    temp = datos[
        ["fecha", "atenciones"]
    ].copy()

    temp["anio"] = anio
    temp["prediccion"] = y_pred
    temp["error_abs"] = abs(
        temp["atenciones"] - temp["prediccion"]
    )

    predicciones_produccion.append(temp)


metricas_produccion = pd.DataFrame(
    metricas_produccion
)

predicciones_produccion = pd.concat(
    predicciones_produccion,
    ignore_index=True
)

display(metricas_produccion)

Modelo seleccionado:
Regresión Lineal


,anio,MAE_modelo,RMSE_modelo,MAE_baseline
0,2020,27931.439884,33963.021670,10429.692308
1,2021,16915.790643,18783.583785,7306.865385
2,2022,16301.460626,20668.770170,19498.673077
3,2023,14787.118032,18962.211528,15496.584906


### 3.4 Análisis de errores y comportamiento del modelo

Durante la validación temporal realizada sobre el período de referencia 2018–2019, el **baseline basado en la demanda de la semana anterior (`lag_1`) presentó el menor MAE promedio (13.378 atenciones)**, superando a los modelos de Machine Learning evaluados. Entre estos últimos, la **Regresión Lineal obtuvo el mejor desempeño**, con un MAE promedio de aproximadamente **15.962 atenciones**, por lo que fue seleccionada como modelo predictivo para las siguientes etapas del proyecto.

Al evaluar el modelo sobre las ventanas de producción simulada se observó un cambio importante en su comportamiento. Durante **2020**, el MAE aumentó hasta aproximadamente **27.931 atenciones**, representando un deterioro considerable respecto del error observado durante el período de referencia. Este resultado coincide temporalmente con el fuerte cambio observado en la serie histórica durante el inicio de la pandemia.

En los años posteriores, el error disminuyó progresivamente: aproximadamente **16.916 en 2021**, **16.301 en 2022** y **14.787 en 2023**, acercándose nuevamente a los niveles observados durante el entrenamiento.

También se observa que el baseline continúa siendo especialmente competitivo en 2020 y 2021, mientras que la Regresión Lineal logra superarlo durante 2022 y 2023. Este comportamiento evidencia que un modelo más complejo no necesariamente supera siempre a una estrategia simple, por lo que el baseline se mantendrá como referencia durante el monitoreo.

Estos resultados sugieren que el mayor deterioro predictivo ocurre durante el período de mayor alteración de la demanda. Sin embargo, el aumento del error por sí solo no permite determinar si los datos de entrada cambiaron respecto del período de entrenamiento. Por esta razón, la siguiente etapa analizará explícitamente la presencia de **data drift** en las variables utilizadas por el modelo.


## 4. Monitoreo del desempeño y Data Drift

Una vez entrenado el modelo con el período de referencia 2018–2019, se evalúa su
comportamiento en las ventanas de producción correspondientes a 2020–2023.

El monitoreo considera dos aspectos complementarios:

1. **Performance del modelo:** se analiza si el error predictivo aumenta respecto
   del período de referencia.
2. **Data drift:** se evalúa si las variables que recibe el modelo presentan una
   distribución diferente de aquella observada durante el entrenamiento.

Esta distinción permite determinar si un deterioro en las predicciones podría estar
asociado a cambios en los datos de producción.

In [ ]:
# 30
# Evaluación del cambio en el desempeño del modelo

mae_referencia = resumen_cv.loc[
    resumen_cv["modelo"] == "Regresión Lineal",
    "MAE_promedio"
].iloc[0]

performance_drift = metricas_produccion.copy()

performance_drift["MAE_referencia"] = mae_referencia

performance_drift["variacion_MAE_pct"] = (
    (
        performance_drift["MAE_modelo"]
        - performance_drift["MAE_referencia"]
    )
    / performance_drift["MAE_referencia"]
    * 100
)

performance_drift["estado_performance"] = np.where(
    performance_drift["variacion_MAE_pct"] > 20,
    "⚠️ Deterioro",
    "✅ Estable"
)

display(
    performance_drift[
        [
            "anio",
            "MAE_referencia",
            "MAE_modelo",
            "variacion_MAE_pct",
            "estado_performance"
        ]
    ]
)

**Interpretación del monitoreo de desempeño:**  
El modelo presenta un deterioro significativo durante 2020, con un aumento aproximado
del 75 % en el MAE respecto del período de referencia 2018–2019.

En 2021 y 2022 el error vuelve a valores cercanos a los observados durante el
entrenamiento, mientras que en 2023 incluso presenta un MAE inferior al valor de
referencia.

Este resultado indica que el mayor deterioro predictivo se concentra en 2020. Sin
embargo, para determinar si este comportamiento está asociado a cambios en los datos
de entrada, es necesario analizar directamente la presencia de data drift.

### 4.1 Detección de Data Drift mediante PSI

Para evaluar si las variables recibidas por el modelo en producción mantienen una
distribución similar a la observada durante el entrenamiento, se utilizará el
**Population Stability Index (PSI)**.

El período 2018–2019 será considerado como distribución de referencia y cada año
entre 2020 y 2023 será evaluado como una ventana independiente de producción.

Se utilizarán los siguientes criterios de interpretación:

- **PSI < 0.10:** distribución estable.
- **PSI entre 0.10 y 0.20:** cambio moderado.
- **PSI > 0.20:** drift significativo.

El análisis se realizará inicialmente sobre las variables numéricas derivadas de la
demanda histórica utilizadas por el modelo.

Las variables de calendario (`mes` y `semana_anio`) no se incluyen en el cálculo de PSI, debido a que su distribución está determinada por la estructura temporal del calendario y no representa un cambio en el comportamiento de la demanda.


In [ ]:
# 31
# Función para calcular Population Stability Index (PSI)

def calcular_psi(referencia, produccion, bins=10):

    referencia = pd.Series(referencia).dropna()
    produccion = pd.Series(produccion).dropna()

    # Crear límites utilizando cuantiles de los datos de referencia
    limites = np.unique(
        np.quantile(
            referencia,
            np.linspace(0, 1, bins + 1)
        )
    )

    # Extender límites para incluir todos los valores de producción
    limites[0] = -np.inf
    limites[-1] = np.inf

    ref_dist = pd.cut(
        referencia,
        bins=limites,
        include_lowest=True
    ).value_counts(normalize=True, sort=False)

    prod_dist = pd.cut(
        produccion,
        bins=limites,
        include_lowest=True
    ).value_counts(normalize=True, sort=False)

    # Evitar divisiones por cero
    epsilon = 1e-6

    ref_dist = np.clip(ref_dist.values, epsilon, None)
    prod_dist = np.clip(prod_dist.values, epsilon, None)

    psi = np.sum(
        (prod_dist - ref_dist)
        * np.log(prod_dist / ref_dist)
    )

    return psi

In [ ]:
# 32
# Cálculo de PSI para las ventanas de producción 2020–2023

variables_drift = [
    "lag_1",
    "lag_2",
    "lag_4",
    "media_4_sem"
]

resultados_psi = []

for anio, datos in ventanas.items():

    for variable in variables_drift:

        psi = calcular_psi(
            train[variable],
            datos[variable]
        )

        if psi < 0.10:
            estado = "✅ Estable"

        elif psi <= 0.20:
            estado = "⚠️ Moderado"

        else:
            estado = "🔴 Drift"

        resultados_psi.append({
            "anio": anio,
            "variable": variable,
            "PSI": psi,
            "estado": estado
        })

resultados_psi = pd.DataFrame(resultados_psi)

display(
    resultados_psi.round({
        "PSI": 3
    })
)

### 4.2 Validación del cambio de distribución

Los valores de PSI obtenidos indican cambios importantes en las variables de demanda
histórica respecto del período de referencia. Para validar que este resultado corresponde
a diferencias reales en los datos y no a un artefacto del cálculo, se compararán
estadísticos descriptivos de las principales variables entre las ventanas temporales.

In [ ]:
# 33
# Comparación de estadísticos por ventana temporal

resumen_distribuciones = []

periodos = {
    "Referencia 2018-2019": train,
    "2020": prod_2020,
    "2021": prod_2021,
    "2022": prod_2022,
    "2023": prod_2023
}

for periodo, datos in periodos.items():

    resumen_distribuciones.append({
        "periodo": periodo,

        "lag_1_media": datos["lag_1"].mean(),
        "lag_1_mediana": datos["lag_1"].median(),

        "media_4_sem_media": datos["media_4_sem"].mean(),
        "media_4_sem_mediana": datos["media_4_sem"].median()
    })

resumen_distribuciones = pd.DataFrame(
    resumen_distribuciones
)

display(
    resumen_distribuciones.round(0)
)

**Interpretación:**  
La comparación de los estadísticos descriptivos confirma cambios importantes en la
distribución de la demanda respecto del período de referencia 2018–2019.

Durante 2020 se observa una disminución marcada en los niveles de demanda histórica,
seguida de una recuperación durante 2021. En 2022 y 2023 la demanda alcanza niveles
superiores a los observados durante el entrenamiento.

Estos resultados respaldan los valores de PSI obtenidos y confirman la presencia de
data drift en las variables utilizadas por el modelo.

In [ ]:
# 34
# Resumen conjunto de Data Drift y desempeño del modelo

# Resumen del PSI por año
resumen_psi = (
    resultados_psi
    .groupby("anio")
    .agg(
        PSI_promedio=("PSI", "mean"),
        PSI_maximo=("PSI", "max"),
        variables_con_drift=(
            "estado",
            lambda x: (x == "🔴 Drift").sum()
        )
    )
    .reset_index()
)

# Integrar con las métricas de performance
monitoreo = performance_drift.merge(
    resumen_psi,
    on="anio",
    how="left"
)

# Definir estado general del sistema
def clasificar_estado(fila):

    if (
        fila["PSI_maximo"] > 0.20
        and fila["variacion_MAE_pct"] > 20
    ):
        return "🚨 Crítico"

    elif fila["PSI_maximo"] > 0.20:
        return "⚠️ Drift"

    else:
        return "✅ Estable"


monitoreo["estado_general"] = monitoreo.apply(
    clasificar_estado,
    axis=1
)

display(
    monitoreo[
        [
            "anio",
            "PSI_promedio",
            "PSI_maximo",
            "variables_con_drift",
            "variacion_MAE_pct",
            "estado_general"
        ]
    ].round(2)
)

**Interpretación del monitoreo:**  
Todas las ventanas de producción presentan cambios significativos en la distribución
de las variables respecto del período de referencia 2018–2019.

Sin embargo, únicamente 2020 combina un nivel elevado de data drift con un deterioro
superior al 20 % en el MAE del modelo, por lo que se clasifica como un estado crítico.

Durante 2021–2023 se mantiene la presencia de drift, pero sin un deterioro importante
del desempeño predictivo. Estos períodos requieren monitoreo, pero no necesariamente
un reentrenamiento inmediato del modelo.

### Consideración sobre las ventanas de producción

Las ventanas 2020–2023 corresponden a períodos cronológicos reales y no a una
simulación artificial de drift creciente.

Por esta razón, la magnitud del PSI no aumenta de forma monotónica. El año 2020
representa un cambio abrupto asociado al período pandémico, seguido por una
recuperación parcial y posteriormente por nuevos niveles de demanda.

Este comportamiento permite evaluar el sistema de monitoreo frente a distintos
niveles reales de cambio en la distribución de los datos.

## 5. Plan de acción ante Data Drift

El sistema utilizará conjuntamente el nivel de data drift y el cambio en el desempeño
del modelo para determinar la acción recomendada.

- **Estable:** continuar operación normal.
- **Drift sin deterioro significativo:** mantener el modelo y aumentar el monitoreo.
- **Drift + deterioro del MAE superior al 20 %:** generar alerta crítica y activar
  revisión para posible reentrenamiento.

El reentrenamiento será semi-automático: el sistema generará un gatillo de alerta,
pero la decisión final de actualizar el modelo requerirá validación previa de los datos.

In [ ]:
# 35
# Definición del plan de acción según el estado del monitoreo

def definir_accion(fila):

    if fila["estado_general"] == "🚨 Crítico":
        return "Revisar datos y activar reentrenamiento"

    elif fila["estado_general"] == "⚠️ Drift":
        return "Mantener modelo y aumentar monitoreo"

    else:
        return "Operación normal"


monitoreo["accion"] = monitoreo.apply(
    definir_accion,
    axis=1
)

display(
    monitoreo[
        [
            "anio",
            "PSI_maximo",
            "variacion_MAE_pct",
            "estado_general",
            "accion"
        ]
    ].round(2)
)

## 6. Conclusiones del análisis

El análisis permitió construir una serie semanal reproducible de atenciones de
urgencia para Chile durante el período 2018–2023 y entrenar un modelo predictivo
utilizando exclusivamente información disponible en semanas anteriores.

El monitoreo mostró que todas las ventanas posteriores al período de referencia
2018–2019 presentan Data Drift. Sin embargo, el deterioro significativo del
desempeño se concentró principalmente en 2020, cuando el MAE aumentó cerca de un
75 % respecto de la referencia.

Este resultado muestra que la presencia de Data Drift no implica necesariamente
un deterioro inmediato del modelo. Por esta razón, la estrategia de monitoreo
combina cambios en la distribución de las variables con cambios en el desempeño
predictivo antes de recomendar una revisión para reentrenamiento.

Los resultados generados en este notebook alimentan posteriormente el servicio
de inferencia mediante FastAPI y el dashboard de monitoreo desarrollado en
Streamlit.

## 7. Generación y exportación de artefactos MLOps

Una vez finalizado el análisis, entrenamiento y monitoreo del modelo, se generan los
artefactos necesarios para implementar el sistema fuera del notebook.

En esta etapa se almacenan el modelo entrenado, la definición de las variables de
entrada y los principales resultados obtenidos durante el monitoreo. Estos archivos
serán utilizados posteriormente por los componentes operativos del proyecto:

- **FastAPI:** servicio local de inferencia para realizar nuevas predicciones.
- **Streamlit:** dashboard de monitoreo de desempeño y Data Drift.
- **Hugging Face:** almacenamiento de los datasets procesados y resultados utilizados
  por el dashboard.
- **GitHub:** repositorio del código necesario para reproducir y ejecutar el sistema.

La separación entre el notebook de análisis y los artefactos de producción permite
que el modelo pueda ser utilizado sin necesidad de ejecutar nuevamente todo el proceso
de exploración, limpieza y entrenamiento.

In [ ]:
# 36
# Generación de artefactos para la implementación MLOps

import joblib
import json

ruta_salida = "/content/mlops_artifacts"
os.makedirs(ruta_salida, exist_ok=True)

# Modelo entrenado
joblib.dump(
    mejor_modelo,
    os.path.join(ruta_salida, "modelo_urgencias.joblib")
)

# Variables utilizadas por el modelo
with open(
    os.path.join(ruta_salida, "features.json"),
    "w"
) as archivo:

    json.dump(features, archivo)

# Resultados de monitoreo
monitoreo.to_csv(
    os.path.join(ruta_salida, "monitoreo.csv"),
    index=False
)

# PSI por variable
resultados_psi.to_csv(
    os.path.join(ruta_salida, "drift_por_variable.csv"),
    index=False
)

# Predicciones
predicciones_produccion.to_csv(
    os.path.join(ruta_salida, "predicciones.csv"),
    index=False
)

# Dataset procesado semanal
serie_semanal.to_csv(
    os.path.join(ruta_salida, "urgencias_semanales_2018_2023.csv"),
    index=False
)

print("✅ Archivos generados:\n")

for archivo in os.listdir(ruta_salida):
    print(archivo)


In [ ]:
# 37
# Comprimir y respaldar los archivos generados

import shutil

shutil.make_archive(
    "/content/mlops_artifacts",
    "zip",
    "/content/mlops_artifacts"
)

print("✅ Archivo creado: /content/mlops_artifacts.zip")

In [ ]:
# 38
# Descargar respaldo de los artefactos al computador

from google.colab import files

files.download("/content/mlops_artifacts.zip")


## 8. Dashboard de monitoreo

El dashboard de monitoreo fue implementado de forma independiente mediante
**Streamlit**, utilizando los resultados procesados almacenados en Hugging Face.

La aplicación permite seleccionar cada ventana de producción (2020–2023) y
visualizar:

- estado general del sistema;
- PSI máximo y Data Drift por variable;
- variación del MAE respecto del período de referencia;
- comparación entre predicción y demanda real;
- evolución histórica de las atenciones de urgencia;
- acción recomendada ante la detección de drift.

El dashboard se ejecuta localmente y su código se encuentra disponible en el
archivo `dashboard.py` del repositorio GitHub del proyecto.